# Prompt caching in the Anthropic & OpenAI APIs

How prompt caching actually works in the two APIs the PXI agent talks to, demonstrated with
Pydantic AI and **verified against the wire**: every claim below is proven by either

1. capturing the exact HTTP request body the SDK sends (via `httpx` event hooks), or
2. reading the cache counters the API returns in `usage`.

**Why this matters for PXI.** Phoenix's agent enables Anthropic caching through
[`AnthropicPromptCacheCapability`](../src/phoenix/server/agents/capabilities/anthropic_prompt_cache.py),
which sets exactly three Pydantic AI settings:

```python
AnthropicModelSettings(
    anthropic_cache=True,                   # top-level auto breakpoint (multi-turn)
    anthropic_cache_tool_definitions=True,  # cache_control on the LAST tool
    anthropic_cache_instructions=True,      # cache_control on the LAST system block
)
```

No equivalent capability exists for OpenAI — and by the end of this notebook you'll see on the
wire why: **Anthropic caching is explicit opt-in via `cache_control` markers; OpenAI caching is
automatic and server-side.**

**To run:** put `ANTHROPIC_API_KEY` and `OPENAI_API_KEY` in the environment and *Run All*.
Makes ~20 small requests to `claude-sonnet-4-5` and `gpt-4o-mini` (≈ $0.15, ≈ 3 minutes).

## The one rule

Both providers cache a **prefix** of the fully rendered request. Any byte change at position
*N* invalidates the cache at every position ≥ *N*. So the only questions that matter are:

1. **In what order are the request parts rendered?** (that order decides what a change destroys)
2. **How do you opt in, and how do you observe hits?**

| | Anthropic | OpenAI |
|---|---|---|
| Opt-in | explicit `cache_control` markers | automatic, nothing to send |
| Raw usage | `cache_creation_input_tokens`, `cache_read_input_tokens` | `cached_tokens` |
| Pydantic AI | `usage.cache_write_tokens`, `usage.cache_read_tokens` | `usage.cache_read_tokens` |

Every experiment below uses the same shape: an agent with **three tools** (long descriptions),
a **long system prompt**, and a short user message — then we perturb one part at a time and
watch how much of the cache survives.

In [ ]:
import asyncio
import json
import os
import uuid
from typing import Any

from pydantic_ai import Agent
from pydantic_ai.tools import Tool

assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in the environment"
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in the environment"

ANTHROPIC_MODEL = "claude-sonnet-4-5"  # min cacheable prefix: 1024 tokens (Opus 4.x: 4096)
OPENAI_MODEL = "gpt-4o-mini"  # min cacheable prefix: 1024 tokens


# Both caches live for ~5 minutes, so a rerun of this notebook could hit stale entries from
# the previous run. Each experiment therefore puts a fresh nonce at the very FRONT of the
# request (in the first tool's description) — one different byte at position 0 isolates the
# whole prefix.
def fresh_nonce() -> str:
    return uuid.uuid4().hex


def pad(label: str, n: int) -> str:
    """~23 tokens per note; sizes each prompt part well past the 1024-token cache minimum."""
    return " ".join(
        f"{label} note {i:03d}: prefer SI units, cite governing equations, "
        "state assumptions explicitly, and show all intermediate steps."
        for i in range(n)
    )


# --- wire capture: record every HTTP request body the SDKs send -------------------------
captured: list[dict[str, Any]] = []


async def capture_request(request: Any) -> None:  # httpx event hook
    captured.append(
        {
            "url": str(request.url),
            "headers": {
                k: v
                for k, v in request.headers.items()
                if k in ("content-type", "anthropic-version", "anthropic-beta")
            },
            "body": json.loads(request.content) if request.content else None,
        }
    )


def truncate(obj: Any, max_len: int = 80) -> Any:
    if isinstance(obj, str) and len(obj) > max_len:
        return obj[:max_len] + f"… [+{len(obj) - max_len} chars]"
    if isinstance(obj, dict):
        return {k: truncate(v, max_len) for k, v in obj.items()}
    if isinstance(obj, list):
        return [truncate(v, max_len) for v in obj]
    return obj


def show_request(req: dict[str, Any]) -> None:
    print("POST", req["url"], "\nheaders:", req["headers"])
    print(json.dumps(truncate(req["body"]), indent=2))

In [ ]:
# The same three tools are used for both providers. Descriptions are padded so the tools
# alone exceed the 1024-token cache minimum. `nonce` varies the FIRST tool (= the very
# start of the request); `last_tool_note` varies the LAST tool. Everything else stays
# byte-identical.
def make_tools(nonce: str, last_tool_note: str = "v1") -> list[Tool[None]]:
    def get_weather(city: str) -> str:
        return f"Sunny in {city}"

    def get_time(timezone: str) -> str:
        return f"12:00 in {timezone}"

    def get_stock_price(ticker: str) -> str:
        return f"{ticker}: $100"

    return [
        Tool(get_weather, description=f"nonce:{nonce} Get current weather. {pad('w', 60)}"),
        Tool(get_time, description=f"Get current time. {pad('t', 30)}"),
        Tool(get_stock_price, description=f"Get stock price. {last_tool_note} {pad('s', 20)}"),
    ]


SYSTEM = f"You are a concise assistant. {pad('style', 50)}"

## Part 1 — Anthropic: what PXI actually sends

Build an agent with the exact three settings `AnthropicPromptCacheCapability` sets, then look
at the raw request body.

In [ ]:
from anthropic import DefaultAsyncHttpxClient
from pydantic_ai.models.anthropic import AnthropicModel, AnthropicModelSettings
from pydantic_ai.providers.anthropic import AnthropicProvider

PXI_SETTINGS = AnthropicModelSettings(
    anthropic_cache=True,
    anthropic_cache_tool_definitions=True,
    anthropic_cache_instructions=True,
)


def make_anthropic_agent(
    nonce: str,
    system: str = SYSTEM,
    last_tool_note: str = "v1",
    settings: AnthropicModelSettings = PXI_SETTINGS,
) -> Agent[None, str]:
    http_client = DefaultAsyncHttpxClient(event_hooks={"request": [capture_request]})
    model = AnthropicModel(ANTHROPIC_MODEL, provider=AnthropicProvider(http_client=http_client))
    return Agent(
        model,
        instructions=system,
        tools=make_tools(nonce, last_tool_note),
        model_settings=settings,
    )


PART1_NONCE = fresh_nonce()
agent = make_anthropic_agent(PART1_NONCE)
result_1 = await agent.run("What is 2+2? Answer in one word.")
print(result_1.output, "\n")
print(result_1.usage)

In [ ]:
# The exact bytes that went over the wire:
show_request(captured[-1])

Three `cache_control` markers, exactly one per setting — and nothing else is cache-related:

* `tools[-1].cache_control` ← `anthropic_cache_tool_definitions` (a breakpoint *after all tools*)
* `system[-1].cache_control` ← `anthropic_cache_instructions` (a breakpoint *after system*)
* top-level `cache_control` ← `anthropic_cache` (the server auto-places a breakpoint on the last
  cacheable block and moves it forward as the conversation grows — this is the multi-turn one)

Each marker says "snapshot the prefix up to here". Anthropic allows at most 4 per request.
The assertions below prove the markers sit where claimed:

In [ ]:
body = captured[-1]["body"]

assert body["cache_control"] == {"type": "ephemeral", "ttl": "5m"}, "top-level auto breakpoint"
assert "cache_control" in body["tools"][-1], "breakpoint on the LAST tool only"
assert all("cache_control" not in t for t in body["tools"][:-1])
assert "cache_control" in body["system"][-1], "breakpoint on the LAST system block"

u1 = result_1.usage
assert u1.cache_write_tokens > 0 and u1.cache_read_tokens == 0, "first request: pure cache write"
print(
    f"turn 1: wrote {u1.cache_write_tokens} tokens to cache, read 0, "
    f"only {u1.details['input_tokens']} tokens billed at the full input rate"
)

In [ ]:
# Turn 2 of the same conversation: the entire turn-1 prompt (tools + system + messages,
# thanks to the auto breakpoint) is served from cache.
result_2 = await agent.run("And 3+3? Answer in one word.", message_history=result_1.all_messages())
u2 = result_2.usage
print(u2)

assert u2.cache_read_tokens == u1.cache_write_tokens, "turn 2 reads exactly what turn 1 wrote"
print(
    f"\nturn 2: read {u2.cache_read_tokens} tokens from cache (~0.1x price), "
    f"wrote {u2.cache_write_tokens} new ones (turn 1's answer + the new question)"
)

**Economics:** cache writes bill at 1.25× the input rate (2× for 1h TTL), reads at ~0.1×.
Two requests already break even; every extra turn of an agent loop is ~90% off on the prefix.

## Part 2 — Anthropic: proving the render order is `tools → system → messages`

Anthropic documents that the request is rendered `tools → system → messages` and that caching
is a prefix match. If that's true, then with breakpoints after *tools* and after *system*:

| change…            | should read from cache…                              |
|--------------------|------------------------------------------------------|
| nothing            | the full prefix (both breakpoints match)             |
| the system prompt  | **only the tools prefix** (tools breakpoint matches) |
| a tool             | **nothing** (tools are position 0)                   |
| the user message   | the full prefix (messages come last)                 |

Five fresh single-turn conversations, perturbing one part at a time. `anthropic_cache` is off
here so the only read points are the two explicit breakpoints, and a fresh nonce isolates this
experiment from Part 1's cache entries:

In [ ]:
EXPLICIT_ONLY = AnthropicModelSettings(
    anthropic_cache_tool_definitions=True,
    anthropic_cache_instructions=True,
)
SYSTEM_B = f"You are a verbose assistant. {pad('style', 50)}"

PART2_NONCE = fresh_nonce()
q = "What is 5+5? Answer in one word."
runs = {
    "A cold (writes cache)": dict(system=SYSTEM, last_tool_note="v1", prompt=q),
    "B identical": dict(system=SYSTEM, last_tool_note="v1", prompt=q),
    "C system changed": dict(system=SYSTEM_B, last_tool_note="v1", prompt=q),
    "D last tool changed": dict(system=SYSTEM, last_tool_note="v2", prompt=q),
    "E user message changed": dict(
        system=SYSTEM, last_tool_note="v1", prompt="What is 7+7? Answer in one word."
    ),
}
stats = {}
for name, spec in runs.items():
    agent = make_anthropic_agent(PART2_NONCE, spec["system"], spec["last_tool_note"], EXPLICIT_ONLY)
    u = (await agent.run(spec["prompt"])).usage
    stats[name] = (u.cache_write_tokens, u.cache_read_tokens)
    print(f"{name:24s} cache_write={u.cache_write_tokens:5d}  cache_read={u.cache_read_tokens:5d}")

In [ ]:
(write_a, read_a) = stats["A cold (writes cache)"]
(_, read_b) = stats["B identical"]
(write_c, read_c) = stats["C system changed"]
(_, read_d) = stats["D last tool changed"]
(_, read_e) = stats["E user message changed"]

assert read_a == 0 and write_a > 0, "cold run: pure write"
assert read_b == write_a, "identical request reads back the whole prefix"
assert 0 < read_c < read_b, "system change: tools prefix SURVIVES -> tools render before system"
assert read_d == 0, "tool change: NOTHING survives -> tools are at position 0"
assert read_e == read_b, "message change: whole prefix survives -> messages render last"

print(
    f"tools prefix survived: {read_c} tokens read; new system prompt written: {write_c} "
    f"tokens. read + write = {read_c + write_c}, vs {read_b} for the unchanged prefix — "
    "the difference is just the tokenization delta between the two system prompts."
)
print("\nOrder proven: tools -> system -> messages")

Run C is the key one: changing the **system prompt** still read the tools prefix from cache,
while run D shows changing a **tool** invalidates everything — including the byte-identical
system prompt that sits behind it. Note the partition in run C: `cache_read + cache_write`
re-assembles run C's full prefix, token for token — Anthropic reads and writes at exact,
controllable byte boundaries.

Practical consequences, now proven:

* **Never reorder, add, or remove tools mid-session** — it's a full cache wipe.
* A volatile system prompt (timestamps! feature flags!) wastes the system+messages cache
  every turn, but the tool definitions still get their discount — that's why PXI sets *both*
  the tools and the instructions breakpoints instead of relying on one.

## Part 3 — OpenAI: automatic caching, nothing on the wire

PXI builds OpenAI models against the **Responses API** (`openai_api_type: "responses"` is the
default in `model_selection.py`) and configures no caching capability at all. Same agent
shape, same wire capture:

In [ ]:
from openai import DefaultAsyncHttpxClient as OpenAIHttpxClient
from pydantic_ai.models.openai import OpenAIResponsesModel, OpenAIResponsesModelSettings
from pydantic_ai.providers.openai import OpenAIProvider

OAI_NONCE = fresh_nonce()

# The ONLY cache-adjacent knobs OpenAI offers (both optional): a routing hint that improves
# hit rates under load, and a retention policy. Neither marks content for caching.
OPENAI_SETTINGS = OpenAIResponsesModelSettings(openai_prompt_cache_key=f"pxi-tut-{OAI_NONCE}")


def oai_instructions(note: str = "") -> str:
    return f"You are a{note} concise assistant. {pad('style', 100)}"


def make_openai_agent(
    instructions: str = oai_instructions(),
    nonce: str = OAI_NONCE,
    last_tool_note: str = "v1",
) -> Agent[None, str]:
    http_client = OpenAIHttpxClient(event_hooks={"request": [capture_request]})
    model = OpenAIResponsesModel(OPENAI_MODEL, provider=OpenAIProvider(http_client=http_client))
    return Agent(
        model,
        instructions=instructions,
        tools=make_tools(nonce, last_tool_note),
        model_settings=OPENAI_SETTINGS,
    )


oai_result_1 = await make_openai_agent().run("What is 2+2? Answer in one word.")
print(oai_result_1.output, "\n")
print(oai_result_1.usage)

In [ ]:
oai_body = captured[-1]["body"]
show_request(captured[-1])

# Prove there is no cache opt-in anywhere in the request: the only key containing "cache"
# is the optional routing hint.
cache_keys = [k for k in json.dumps(oai_body).split('"') if "cache" in k.lower()]
assert cache_keys == ["prompt_cache_key"], cache_keys
print("\ncache-related content on the wire:", cache_keys, "— caching itself is automatic")

In [ ]:
# An identical request now hits the cache automatically. OpenAI's cache is best-effort
# (a fresh identical request occasionally misses), so retry a couple of times if needed.
async def run_until_cached(make_agent, prompt: str, attempts: int = 4):
    for i in range(attempts):
        r = await make_agent().run(prompt)
        if r.usage.cache_read_tokens > 0 or i == attempts - 1:
            return r
        await asyncio.sleep(2)


oai_result_2 = await run_until_cached(make_openai_agent, "What is 2+2? Answer in one word.")
u = oai_result_2.usage
print(u)
assert u.cache_read_tokens > 0, "re-run this cell: OpenAI's cache is best-effort"
print(
    f"\n{u.cache_read_tokens} of {u.input_tokens} input tokens served from cache "
    "(cached input is billed at a ~50-90% discount depending on model)"
)

Requirements OpenAI documents (and these runs confirm): prefix must be ≥ 1024 tokens, hits are
reported in 128-token increments, entries live ~5–10 minutes
(`openai_prompt_cache_retention="24h"` extends that). Note `cache_read_tokens` above is *not*
equal to `input_tokens` — the suffix (final message + scaffolding) is never cached.

## Part 4 — OpenAI: probing the order with no markers

We can't place breakpoints, but we can still measure the render order the same way: perturb
one part, and `cached_tokens` reports how much prefix survived. Because hits are best-effort,
each perturbation is probed with three *fresh* variants (so a probe never matches its own
earlier attempt) and we take the max.

In [ ]:
async def probe(name: str, variants: list[dict], prompt="What is 2+2? Answer in one word."):
    vals = []
    for v in variants:
        agent = make_openai_agent(
            instructions=oai_instructions(v.get("instr", "")),
            nonce=v.get("nonce", OAI_NONCE),
            last_tool_note=v.get("tool", "v1"),
        )
        vals.append((await agent.run(v.get("prompt", prompt))).usage.cache_read_tokens)
        await asyncio.sleep(1)
    print(f"{name:22s} cached_tokens={vals}  ->  max {max(vals)}")
    return max(vals)


full = await probe("message changed", [dict(prompt="What is 7+7? Answer in one word.")])
first_tool = await probe("first tool changed", [dict(nonce=f"{OAI_NONCE}-{i}") for i in "BCD"])
last_tool = await probe("last tool changed", [dict(tool=f"v{i}") for i in "234"])
instr = await probe("instructions changed", [dict(instr=f" {w}") for w in ("very", "so", "too")])

In [ ]:
assert full > 0, "message change: the whole fixed prefix survives -> messages last"
assert first_tool == 0, "first-tool change: nothing survives -> tools render FIRST"
assert 0 < instr < full, "instructions change: a chunk (the tools region) survives"
assert last_tool < instr, "changing an earlier byte always kills more cache"
print("Order observed: tools -> instructions -> messages  (same macro order as Anthropic)")

Same macro order as Anthropic — but with two differences worth noticing:

1. **You can't protect the tools prefix precisely.** Anthropic's explicit tool breakpoint made
   Part 2's run C read *exactly* the tools region. OpenAI picks snapshot points server-side:
   the partial hits above are real but land *below* the exact component byte boundaries (in
   our probes, several hundred tokens short — compare `last tool changed` with where the
   third tool actually starts). Treat it as a black box: static content early, volatile late.
2. **Everything is best-effort.** No exact read points, occasional misses on identical
   requests — which is why these assertions use max-over-fresh-variants while the Anthropic
   ones could assert exact token equalities.

## Summary

| | Anthropic | OpenAI |
|---|---|---|
| Opt-in | explicit `cache_control` markers on the wire | automatic; nothing on the wire |
| Knobs | breakpoint placement + TTL (`5m`/`1h`), max 4 | routing key, retention policy |
| Render order | **tools → system → messages** (exact) | **tools → instructions → messages** |
| Reads happen | exactly at your breakpoints | server-chosen snapshots, 128-token granularity |
| Min prefix | 1024–4096 tokens (model-dependent) | 1024 tokens |
| Pricing | write 1.25× (2× for 1h), read ~0.1× | read discount ~50–90% (model-dependent) |
| Failure mode | forget a marker → **no caching at all** | none to forget |

Which is exactly the PXI design:

* **Anthropic** → `AnthropicPromptCacheCapability` sets all three markers (tools, instructions,
  and the moving multi-turn breakpoint), because without markers Anthropic caches nothing.
* **OpenAI** → no capability, because there is nothing to send.

And the shared rule for both, since tools render first: keep the tool list and system prompt
byte-stable across a session, and put anything volatile in the last user message.